# Export PhysiCell Decay CSVs

This notebook extracts the PhysiCell decay-study outputs using the same microenvironment processing approach as the comparison notebook and writes one CSV into each experiment folder.

Each CSV contains the fields needed later by the plotting workflow: `tool`, `run_id`, `dx_um`, `dt_min`, `average_source`, `center_source`, `resolution_label`, `time_min`, `average_uM`, and `center_uM`.

The `resolution_label` column is kept because the downstream comparison notebooks still use it directly in grouping and plot labels.

In [ ]:
from itertools import product
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from pctk import multicellds

In [ ]:
ROOT = Path('/home/tntiniak/Work/observatory_benchmark')
PHYSICELL_DIR = ROOT / 'PhysiCell' / 'results' / 'decay_study'
OUTPUT_FILENAME = 'physicell_decay_plot_data.csv'
EXPERIMENT_FOLDERS = [
    PHYSICELL_DIR / 'size_5',
    PHYSICELL_DIR / 'size_10',
    PHYSICELL_DIR / 'size_20',
    PHYSICELL_DIR / 'size_40',
]

EXPERIMENT_FOLDERS

In [ ]:
def make_physicell_resolution_label(dx_um: float) -> str:
    return f'voxel size={dx_um:.0f} um'


def load_physicell_run(folder: Path) -> pd.DataFrame:
    size_token = folder.name.split('_')[1]
    settings_path = folder / f'PhysiCell_settings_{size_token}.xml'
    dx_um = float(ET.parse(settings_path).getroot().findtext('.//domain/dx'))

    reader = multicellds.MultiCellDS(output_folder=str(folder))

    average_uM = []
    center_uM = []
    center_indices = None

    for _, microenvironment in reader.microenvironment_as_matrix_iterator():
        concentration_field = microenvironment[4]

        if center_indices is None:
            n_voxels = concentration_field.size
            n_per_dim = int(round(n_voxels ** (1 / 3)))
            dims = (n_per_dim, n_per_dim, n_per_dim)

            midpoints = []
            for dim in dims:
                if dim % 2 == 1:
                    midpoints.append([dim // 2])
                else:
                    midpoints.append([dim // 2 - 1, dim // 2])

            center_coords = list(product(*midpoints))
            center_indices = np.ravel_multi_index(np.array(center_coords).T, dims)

        average_uM.append(concentration_field.mean() / 602.2)
        center_uM.append(concentration_field[center_indices].mean() / 602.2)

    time_min = np.round(np.arange(len(average_uM), dtype=float) * 0.1, 2)
    run_id = f'PhysiCell_dx_{int(dx_um)}um'

    return pd.DataFrame({
        'tool': 'PhysiCell',
        'run_id': run_id,
        'dx_um': dx_um,
        'dt_min': np.nan,
        'average_source': 'mean over all voxels from microenvironment[4]',
        'center_source': 'mean over center voxels from microenvironment[4]',
        'resolution_label': make_physicell_resolution_label(dx_um),
        'time_min': time_min,
        'average_uM': np.array(average_uM, dtype=float),
        'center_uM': np.array(center_uM, dtype=float),
    })

In [ ]:
exported_files = []
preview_tables = []
row_counts = []

for folder in EXPERIMENT_FOLDERS:
    frame = load_physicell_run(folder)
    output_path = folder / OUTPUT_FILENAME
    frame.to_csv(output_path, index=False)
    exported_files.append(output_path)
    preview_tables.append(frame.head(3))
    row_counts.append(len(frame))

pd.DataFrame({
    'experiment_folder': [path.parent.name for path in exported_files],
    'csv_path': [str(path) for path in exported_files],
    'rows_written': row_counts,
})

In [ ]:
pd.concat(preview_tables, ignore_index=True)